In [2]:
%pip install mrmr-selection

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Import neccessary libraries
import itertools

import pandas as pd
import numpy as np

# For graph plot
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns

# For feature scaling and selection
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
from mrmr import mrmr_regression

# For DLNM model
from statsmodels.gam.gam_cross_validation.cross_validators import KFold
import statsmodels.api as sm
from statsmodels.gam.smooth_basis import UnivariateBSplines, UnivariateCubicCyclicSplines
from statsmodels.gam.api import GLMGam, BSplines, CyclicCubicSplines
from statsmodels.gam.gam_cross_validation.gam_cross_validation import MultivariateGAMCVPath
from statsmodels.genmod.families.family import Tweedie
from statsmodels.genmod.families.links import log
from statsmodels.gam.smooth_basis import GenericSmoothers
from pydlnm.basis import CrossBasis
from pydlnm.prediction import crosspred
from pydlnm.utils import logknots
from dlnm import crossbasis, plot_crosspred, cb_pen


In [5]:
def crossbasis_construct(dataframe, col_lst, max_lag, var_degree, var_df, lag_degree, lag_df):
    cb_list = []
    for col in col_lst:
        knots_var = np.quantile(dataframe[col].dropna(), [0.1, 0.75, 0.9])
        lag_knots = logknots([0, max_lag], nk=lag_df)

        tmp_cb = CrossBasis(dataframe[col].values, lag=max_lag, 
                                argvar={"fun": "bs", "degree": var_degree, "df": var_df}, 
                                arglag={"fun": "bs", "degree": lag_degree, "df": lag_df})
        cb_list.append(tmp_cb)

    X_exposure = np.concat(cb_list, axis=1)

    model_data = np.concat([dataframe.drop(columns=["From"]).to_numpy(), X_exposure], axis=1)
    model_data = pd.DataFrame(model_data)
    exp_name_list = [f"{col}_basis_{i}" for col in col_lst for i in range(tmp_cb.shape[1])]
    model_data.columns = [n for n in dataframe.columns if n!="From"] + exp_name_list
    model_data[exp_name_list] = model_data[exp_name_list].fillna(0)
    exp_names = " + ".join(exp_name_list)

    return model_data, exp_names, cb_list

In [26]:
def smoothers_construct(dataframe, bs_dict, cs_dict):
    bs = UnivariateBSplines(dataframe[bs_dict["name"]], df=bs_dict["df"], degree=bs_dict["degree"], include_intercept=False, variable_name=bs_dict["name"])
    cs = UnivariateCubicCyclicSplines(dataframe[cs_dict["name"]], df=cs_dict["df"], constraints="center", variable_name=cs_dict["name"])
    sm_weights = [bs, cs]
    
    X_smooth = dataframe[[bs_dict["name"]]+[cs_dict["name"]]]
    smoothers = GenericSmoothers(X_smooth, sm_weights)
    
    return smoothers

In [7]:
def NB_GAM_model(dataframe, tar_var, exp_names, smoothers, a): 
    formula = f"{tar_var} ~ {exp_names}"

    NegativeBinomial_gam_model = GLMGam.from_formula(formula, data=dataframe, smoother=smoothers, alpha=a, family=sm.families.NegativeBinomial(alpha=0.5))
    result = NegativeBinomial_gam_model.fit()
    # result = NegativeBinomial_gam_model.fit_regularized(method="l1_slsqp", alpha=1.0)
        # no penalization for Negative Binomial

    return NegativeBinomial_gam_model, result

In [8]:
def Tweddie_GAM_model(dataframe, tar_var, exp_names, smoothers, a):
    formula = f"{tar_var} ~ {exp_names}"
    Tweddie_gam_model = GLMGam.from_formula(formula, data=dataframe, smoother=smoothers, alpha=a, family=Tweedie(link=log(), var_power=1.5))
    result = Tweddie_gam_model.fit()

    return Tweddie_gam_model, result

In [9]:
def pred_cal(result, dataframe, cb_lst, mete_lst, i):
    n_cb_cols = cb_lst[i].shape[1]
    start_idx = 0
    end_idx = start_idx+n_cb_cols
    cb_coef = result.params.values[start_idx:end_idx]
    cb_vcov = result.cov_params().iloc[start_idx:end_idx, start_idx:end_idx]
    at = np.linspace(dataframe[mete_lst[i]].min(), dataframe[mete_lst[i]].max(), 100)
    cen = dataframe[mete_lst[i]].median()
    pred = crosspred(basis=cb_lst[i], coef=cb_coef, vcov=cb_vcov, model_link="log", at=at, cen=cen)
    
    print(f"{mete_lst[i]} RR range: {pred.allRRfit.min():.3f} to {pred.allRRfit.max():.3f}")

    return pred

In [10]:
def graph_3Dplot(pred, exp):
    fig = plot_crosspred(pred, ptype="3d")

    ax = fig.gca()
    ax.set_title("3D Exposure-Lag-Response Association Surface", fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel(exp, fontsize=12, labelpad=10)
    ax.set_ylabel("Lag (Weeks)", fontsize=12, labelpad=10)
    ax.set_zlabel("Residual Risk (RR)", labelpad=10)
    ax.set_yticks(np.linspace(0, 4, pred.matRRfit.shape[1]))

    plt.tight_layout()
    plt.show()

In [11]:
def graph_expresponseplot(pred, exp):
    fig = plot_crosspred(pred, ptype="overall")
    cen = df[exp].median()

    ax = fig.gca()

    ax.axvline(x=cen, color='black', linestyle=':', linewidth=1.2, label=f'Centering Temp ({cen:.1f}°C)')
    ax.set_title('Overall Cumulative Exposure-Response Association', fontsize=14, pad=15)
    ax.set_xlabel(exp, fontsize=12)
    ax.legend(loc="best", frameon=True)

    plt.show()

In [12]:
def graph_lagresponseplot(pred, target_var):
    fig = plot_crosspred(
        pred, 
        ptype="slices", 
        var=target_var
    )

    ax = fig.gca()

    ax.set_title(f"Lag-Response Curve at {target_var}", fontsize=14, fontweight='bold')
    ax.set_xlabel("Lag (Weeks)", fontsize=12)
    ax.set_ylabel("Relative Risk (RR)", fontsize=12)
    ax.axhline(1, color='gray', linestyle='--', alpha=0.7) # Add baseline at RR=1
    
    plt.show()

In [13]:
def graph_plot(pred, exp):
    graph_3Dplot(pred, exp)
    graph_expresponseplot(pred, exp)
    graph_lagresponseplot(pred, 15)

In [14]:
def alpha_byAIC(dataframe, tar_var, exp_names, smoothers, alpha):
    best_a = 0
    best_aic = np.inf

    for a1, a2 in itertools.product(alpha, alpha):
        model, result = NB_GAM_model(dataframe, tar_var, exp_names, smoothers, [a1, a2])
        print(f"alpha {[a1, a2]} with AIC value {result.aic}")
        if result.aic < best_aic:
            best_aic = result.aic
            best_a = [a1, a2]

    return best_a

In [15]:
def alpha_byGCV(dataframe, tar_var, exp_names, smoothers):
    model, result = NB_GAM_model(dataframe, tar_var, exp_names, smoothers, [100, 500])
    best_a = model.select_penweight(criterion="gcv")
    return best_a[0]

In [16]:
def feature_scaling():
    df[mete_col] = StandardScaler().fit_transform(df[mete_col])
    df["Year"] = df["Year"] - df["Year"].mean()
  

In [17]:
def feature_selection_MI(dataframe, exp_col, outcome_col):
    X = dataframe[exp_col]
    Y = dataframe[outcome_col]

            # Calculate mutual information for regression
    mi_scores = mutual_info_regression(X, Y)

            # Display scores
    mi_df = pd.DataFrame({'Feature': [col for col in X], 'MI_Score': mi_scores})
    selected_features = mi_df.loc[mi_df["MI_Score"]>=0, "Feature"].to_list()

    return mi_df.sort_values(by='MI_Score', ascending=False), selected_features

In [18]:
def feature_selection_MRMR(dataframe, exp_col, outcome_col):
    selected_features = mrmr_regression(X=dataframe[exp_col], y=dataframe[outcome_col], K=5)
    print("Selected Features:", selected_features)
            
    return selected_features 

---

In [19]:
# Import data and basic settings
df = pd.read_csv("data.csv")

df.columns = ["Year", "Week", "From", "temprange", "temp", "RH", "SR", "AP", "DPT", "WS", "UV", "H1", "H3", "B", "AandB"]

mete_col = ["temprange", "temp", "RH", "SR", "AP", "DPT", "WS", "UV"]
flu_col = ["H1", "H3", "B", "AandB"]


In [20]:
# Feature scaling and selection
np.random.seed(42)

feature_scaling()
feature_selection_MI(df, mete_col, "H1")
mete_col = feature_selection_MRMR(df, mete_col, "H1")

100%|██████████| 5/5 [00:04<00:00,  1.17it/s]

Selected Features: ['UV', 'RH', 'WS', 'temp', 'SR']


In [29]:
model_data, exp_names, cb_list = crossbasis_construct(df, mete_col, 4, 2, 5, 3, 6)
smoothers = smoothers_construct(df, {"name": "Year", "degree": 2, "df": 3}, {"name": "Week", "df": 4})

In [27]:
alpha_choice = [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]
best_a_aic = alpha_byAIC(df, "H1", " + ".join(mete_col), smoothers, alpha_choice)

print(f"alpha {best_a_aic} is the best tested with AIC")

alpha [0.01, 0.01] with AIC value 14661.647525055321
alpha [0.01, 0.1] with AIC value 14661.647469220929
alpha [0.01, 1.0] with AIC value 14661.646915997322
alpha [0.01, 10.0] with AIC value 14661.641805078798
alpha [0.01, 50.0] with AIC value 14661.62835964347
alpha [0.01, 100.0] with AIC value 14661.632072920896
alpha [0.1, 0.01] with AIC value 14661.647517549167
alpha [0.1, 0.1] with AIC value 14661.647461714912
alpha [0.1, 1.0] with AIC value 14661.646908492616
alpha [0.1, 10.0] with AIC value 14661.641797589045
alpha [0.1, 50.0] with AIC value 14661.628352219483
alpha [0.1, 100.0] with AIC value 14661.632065576774
alpha [1.0, 0.01] with AIC value 14661.647442502288
alpha [1.0, 0.1] with AIC value 14661.647387280555
alpha [1.0, 1.0] with AIC value 14661.646833460018
alpha [1.0, 10.0] with AIC value 14661.641722706167
alpha [1.0, 50.0] with AIC value 14661.628277995576
alpha [1.0, 100.0] with AIC value 14661.631992150076
alpha [10.0, 0.01] with AIC value 14661.646694451498
alpha [10

In [35]:
model, result = NB_GAM_model(model_data, "H1", exp_names, smoothers, best_a_aic)
print(result.summary())


c:\Users\yanss\anaconda3\Lib\site-packages\statsmodels\gam\generalized_additive_model.py:1223: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = lm.WLS(aug_y, aug_x, aug_weights).fit()
c:\Users\yanss\anaconda3\Lib\site-packages\statsmodels\gam\generalized_additive_model.py:1223: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = lm.WLS(aug_y, aug_x, aug_weights).fit()
c:\Users\yanss\anaconda3\Lib\site-packages\statsmodels\gam\generalized_additive_model.py:1223: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = lm.WLS(aug_y, aug_x, aug_weights).fit()
c:\Users\yanss\anaconda3\Lib\site-packages\statsmodels\gam\generalized_additive_model.py:1223: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_result

                 Generalized Linear Model Regression Results                  
Dep. Variable:                     H1   No. Observations:                 1253
Model:                         GLMGam   Df Residuals:                  1125.88
Model Family:        NegativeBinomial   Df Model:                       126.12
Link Function:                    Log   Scale:                          1.0000
Method:                         PIRLS   Log-Likelihood:                -6786.2
Date:                Mon, 09 Nov 2026   Deviance:                       7415.5
Time:                        16:13:25   Pearson chi2:                 8.96e+03
No. Iterations:                  1000   Pseudo R-squ. (CS):             0.9616
Covariance Type:            nonrobust                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         3.0031      0.483      6.220

c:\Users\yanss\anaconda3\Lib\site-packages\statsmodels\gam\generalized_additive_model.py:1223: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = lm.WLS(aug_y, aug_x, aug_weights).fit()


In [37]:
X = np.asarray(model.exog, dtype=float)

print("Design matrix:", X.shape)
print("Matrix rank:", np.linalg.matrix_rank(X))
print("Number of parameters:", X.shape[1])
print("Condition number:", np.linalg.cond(X))

print("Column names:", model.exog_names)

Design matrix: (1253, 156)
Matrix rank: 131
Number of parameters: 156
Condition number: 1.50276487663046e+18
Column names: ['Intercept', 'UV_basis_0', 'UV_basis_1', 'UV_basis_2', 'UV_basis_3', 'UV_basis_4', 'UV_basis_5', 'UV_basis_6', 'UV_basis_7', 'UV_basis_8', 'UV_basis_9', 'UV_basis_10', 'UV_basis_11', 'UV_basis_12', 'UV_basis_13', 'UV_basis_14', 'UV_basis_15', 'UV_basis_16', 'UV_basis_17', 'UV_basis_18', 'UV_basis_19', 'UV_basis_20', 'UV_basis_21', 'UV_basis_22', 'UV_basis_23', 'UV_basis_24', 'UV_basis_25', 'UV_basis_26', 'UV_basis_27', 'UV_basis_28', 'UV_basis_29', 'RH_basis_0', 'RH_basis_1', 'RH_basis_2', 'RH_basis_3', 'RH_basis_4', 'RH_basis_5', 'RH_basis_6', 'RH_basis_7', 'RH_basis_8', 'RH_basis_9', 'RH_basis_10', 'RH_basis_11', 'RH_basis_12', 'RH_basis_13', 'RH_basis_14', 'RH_basis_15', 'RH_basis_16', 'RH_basis_17', 'RH_basis_18', 'RH_basis_19', 'RH_basis_20', 'RH_basis_21', 'RH_basis_22', 'RH_basis_23', 'RH_basis_24', 'RH_basis_25', 'RH_basis_26', 'RH_basis_27', 'RH_basis_28'

In [38]:

# Working weights from the last PIRLS iteration
w = np.asarray(model.weights)

print("Working weights:")
print("Minimum:", np.min(w))
print("Maximum:", np.max(w))
print("Median:", np.median(w))
print("Number of near-zero weights:", np.sum(w < 1e-12))

# Smoothing penalty matrix
S = model.penal.penalty_matrix(alpha=model.alpha)

print("\nPenalty matrix:")
print("Shape:", S.shape)
print("Maximum absolute value:", np.max(np.abs(S)))

print("Penalty eigenvalues:")
print(np.linalg.eigvalsh((S + S.T) / 2))

Working weights:
Minimum: 0.27153740112518293
Maximum: 1.9968675179350936
Median: 1.8822482625856083
Number of near-zero weights: 0

Penalty matrix:
Shape: (156, 156)
Maximum absolute value: 0.5613592465110554
Penalty eigenvalues:
[0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0

In [39]:
from statsmodels.tools.linalg import matrix_sqrt

X = np.asarray(model.exog)

# Construct the penalty augmentation
R = matrix_sqrt(2 * S)

A = np.vstack([X, R])

# Add the WLS weights
w_aug = np.concatenate([
    w,
    np.ones(R.shape[0])
])

# Weighted augmented design matrix
A_weighted = A * np.sqrt(w_aug)[:, None]

# Singular values
sv = np.linalg.svd(
    A_weighted,
    compute_uv=False
)

print("\nWeighted augmented matrix:")
print("Shape:", A_weighted.shape)
print("Rank:", np.linalg.matrix_rank(A_weighted))
print("Number of columns:", A_weighted.shape[1])
print("Condition number:", np.linalg.cond(A_weighted))
print("Smallest singular values:", sv[-5:])


Weighted augmented matrix:
Shape: (1257, 156)
Rank: 131
Number of columns: 156
Condition number: 4.155579416481688e+17
Smallest singular values: [1.83813173e-15 1.57718168e-15 1.13485045e-15 5.37337306e-16
 2.96824848e-16]


In [ ]:
import numpy as np
import statsmodels.regression.linear_model as lm

# Keep original WLS fit method
original_fit = lm.WLS.fit

def diagnostic_fit(self, *args, **kwargs):

    # This is the actual weighted design matrix
    # used internally by Statsmodels
    X_weighted = np.asarray(self.wexog)

    singular_values = np.linalg.svd(
        X_weighted,
        compute_uv=False
    )

    rank = np.linalg.matrix_rank(X_weighted)

    print("\n========== WLS DIAGNOSTICS ==========")

    print("Matrix shape:", X_weighted.shape)

    print("Rank:", rank)

    print("Number of parameters:", X_weighted.shape[1])

    print("Condition number:", np.linalg.cond(X_weighted))

    print("Smallest singular values:")
    print(singular_values[-5:])

    print("WLS weights:")
    print("Min:", np.min(self.weights))
    print("Max:", np.max(self.weights))

    print("=====================================")

    return original_fit(self, *args, **kwargs)


# Temporarily replace WLS.fit
lm.WLS.fit = diagnostic_fit

try:
    results = model.fit()
finally:
    # Restore original Statsmodels function
    lm.WLS.fit = original_fit

In [41]:
print("Number of smoothers:")
print(len(model.smoother.smoothers))

print("\nSmoother column names:")
print(model.smoother.col_names)

print("\nAlpha values:")
print(model.alpha)


for i, sm in enumerate(model.smoother.smoothers):

    B = np.asarray(sm.basis)
    S = np.asarray(sm.cov_der2)

    print(f"\nSmoother {i}")
    print("Variable:", sm.variable_name)
    print("Basis shape:", B.shape)
    print("Basis rank:", np.linalg.matrix_rank(B))
    print("Penalty shape:", S.shape)
    print("Penalty rank:", np.linalg.matrix_rank(S))

    if i < len(model.alpha):
        print("Alpha:", model.alpha[i])


Number of smoothers:
2

Smoother column names:
['Year_s0', 'Year_s1', 'Week_s0', 'Week_s1', 'Week_s2']

Alpha values:
[100.0, 50.0]

Smoother 0
Variable: Year
Basis shape: (1253, 2)
Basis rank: 2
Penalty shape: (2, 2)
Penalty rank: 1
Alpha: 100.0

Smoother 1
Variable: Week
Basis shape: (1253, 3)
Basis rank: 3
Penalty shape: (3, 3)
Penalty rank: 3
Alpha: 50.0


In [42]:
X = np.asarray(model.exog, dtype=float)

variables = ["UV", "RH", "WS", "temp", "SR"]

for var in variables:

    indices = [
        i for i, name in enumerate(model.exog_names)
        if name.startswith(f"{var}_basis_")
    ]

    B = X[:, indices]

    print(f"\n{var}")
    print("Columns:", B.shape[1])
    print("Rank:", np.linalg.matrix_rank(B))
    print("Unique raw observations:", df[var].nunique())


UV
Columns: 30
Rank: 25
Unique raw observations: 14

RH
Columns: 30
Rank: 25
Unique raw observations: 73

WS
Columns: 30
Rank: 25
Unique raw observations: 160

temp
Columns: 30
Rank: 25
Unique raw observations: 284

SR
Columns: 30
Rank: 25
Unique raw observations: 938


In [44]:


U, s, Vt = np.linalg.svd(
    X,
    full_matrices=False
)

tol = max(X.shape) * np.finfo(float).eps * s[0]

rank = np.sum(s > tol)

# Null-space vectors
null_space = Vt[rank:, :]

# Measure the participation of each column
importance = np.linalg.norm(null_space, axis=0)

redundant_candidates = pd.Series(
    importance,
    index=model.exog_names
).sort_values(ascending=False)

print(redundant_candidates.head(40))

UV_basis_3       0.539796
RH_basis_15      0.539796
UV_basis_21      0.539796
UV_basis_14      0.539796
temp_basis_20    0.539796
temp_basis_15    0.539796
RH_basis_2       0.539796
UV_basis_26      0.539796
SR_basis_3       0.539796
temp_basis_2     0.539796
SR_basis_2       0.539796
temp_basis_26    0.539796
WS_basis_21      0.539796
WS_basis_27      0.539796
SR_basis_15      0.539796
WS_basis_8       0.539796
RH_basis_9       0.539796
SR_basis_27      0.539796
WS_basis_9       0.539796
temp_basis_14    0.539796
RH_basis_20      0.539796
UV_basis_9       0.539796
WS_basis_15      0.539796
WS_basis_3       0.539796
SR_basis_20      0.539796
WS_basis_14      0.539796
RH_basis_21      0.539796
temp_basis_8     0.539796
SR_basis_26      0.539796
temp_basis_3     0.539796
RH_basis_14      0.539796
temp_basis_9     0.539796
SR_basis_21      0.539796
UV_basis_8       0.539796
UV_basis_27      0.539796
SR_basis_14      0.539796
RH_basis_3       0.539796
SR_basis_9       0.539796
SR_basis_8  

In [45]:
for var in ["UV", "RH", "WS", "temp", "SR"]:

    indices = [
        i for i, name in enumerate(model.exog_names)
        if name.startswith(f"{var}_basis_")
    ]

    B = np.asarray(model.exog)[:, indices]

    x = df[var].to_numpy()

    print(f"\n{var}")
    print("Unique predictor values:", np.unique(x).size)
    print("Unique basis rows:", np.unique(B, axis=0).shape[0])
    print("Basis rank:", np.linalg.matrix_rank(B))
    print("Basis columns:", B.shape[1])


UV
Unique predictor values: 14
Unique basis rows: 782
Basis rank: 25
Basis columns: 30

RH
Unique predictor values: 73
Unique basis rows: 1250
Basis rank: 25
Basis columns: 30

WS
Unique predictor values: 160
Unique basis rows: 1250
Basis rank: 25
Basis columns: 30

temp
Unique predictor values: 284
Unique basis rows: 1250
Basis rank: 25
Basis columns: 30

SR
Unique predictor values: 938
Unique basis rows: 1250
Basis rank: 25
Basis columns: 30


In [46]:
import numpy as np
from statsmodels.gam.smooth_basis import UnivariateBSplines

variables = ["UV", "RH", "WS", "temp", "SR"]

for var in variables:

    x = df[var].to_numpy()

    # Use the exact original spline settings here
    bs = UnivariateBSplines(
        x,
        df=df,
        degree=3,
        include_intercept=False,
        variable_name=var
    )

    B_new = np.asarray(bs.basis)

    # Existing basis columns in the fitted model
    indices = [
        i for i, name in enumerate(model.exog_names)
        if name.startswith(f"{var}_basis_")
    ]

    B_old = np.asarray(model.exog)[:, indices]

    print(f"\n{var}")
    print("Unique x:", np.unique(x).size)
    print("Unique rebuilt basis rows:",
          np.unique(B_new, axis=0).shape[0])
    print("Rebuilt basis rank:",
          np.linalg.matrix_rank(B_new))

    print("Old basis shape:", B_old.shape)
    print("New basis shape:", B_new.shape)

    if B_old.shape == B_new.shape:
        print("Old and new bases identical:",
              np.allclose(B_old, B_new))

TypeError: unsupported operand type(s) for -: 'str' and 'int'

In [587]:
best_a_gcv = alpha_byGCV(df, "H1", " + ".join(mete_col), smoothers)

print(f"alpha {best_a_gcv} is the best tested with GCV")


alpha [1.40878725e+43 6.69160853e+04] is the best tested with GCV


In [589]:
model, result = NB_GAM_model(df, "H1", " + ".join(mete_col), smoothers, best_a_gcv)
print(result.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                     H1   No. Observations:                 1253
Model:                         GLMGam   Df Residuals:                     1253
Model Family:        NegativeBinomial   Df Model:                           -1
Link Function:                    Log   Scale:                          1.0000
Method:                         PIRLS   Log-Likelihood:                -94235.
Date:                Mon, 09 Nov 2026   Deviance:                   1.8231e+05
Time:                        09:04:01   Pearson chi2:                 3.38e+07
No. Iterations:                     3   Pseudo R-squ. (CS):         -1.600e+59
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   9.979e-35   5.34e-29   1.87e-06      1.0

In [ ]:
pred = pred_cal(result, df, cb_list, mete_col, 0)
graph_plot(pred, mete_col[0])

ValueError: Variable basis (5) doesn't match coefficients (12)